In [1]:
import torch as t
from torch import nn
from torch.nn import CrossEntropyLoss
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM
from transformers.models.mamba.modeling_mamba import MambaForCausalLM, MambaCache, MambaModel

/storage_1/lruggeri/miniconda3/envs/MambaInterp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Import mamba from HF
model_name = "state-spaces/mamba-130m-hf"

model = MambaForCausalLM.from_pretrained(
    model_name
)
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)
input_ids = tokenizer("Once upon a time", return_tensors="pt")["input_ids"]

out = model.generate(input_ids, max_new_tokens=10)
print(tokenizer.batch_decode(out))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['Once upon a time, there was a man named\n\nBilly']


In [4]:
from mamba_py.mambapy.mamba import MambaForLM, MambaConfig

config = MambaConfig(
    model.config.d_model,
    model.config.n_layer,
    model.config.vocab_size
)

m = MambaForLM(config)

In [6]:
m.load_state_dict(model.state_dict())

<All keys matched successfully>

In [20]:
config = AutoConfig.from_pretrained(model_name)

In [ ]:
# This is how you reinitialize the model
model = AutoModelForCausalLM.from_config(config)
out = model.generate(input_ids, max_new_tokens=10)
print(tokenizer.batch_decode(out))

['Once upon a time time time time time time time time time time time']


In [25]:
from transformers import GenerationConfig

GenerationConfig.from_pretrained(model_name)

GenerationConfig {
  "bos_token_id": 0,
  "eos_token_id": 0,
  "pad_token_id": 0
}

In [20]:
config.n_layer = 12
config.num_hidden_layers = 12
config

NameError: name 'config' is not defined

In [32]:
AutoModelForCausalLM.from_config(config)

MambaForCausalLM(
  (backbone): MambaModel(
    (embeddings): Embedding(50280, 768)
    (layers): ModuleList(
      (0-11): 12 x MambaBlock(
        (norm): MambaRMSNorm(768, eps=1e-05)
        (mixer): MambaMixer(
          (conv1d): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), padding=(3,), groups=1536)
          (act): SiLU()
          (in_proj): Linear(in_features=768, out_features=3072, bias=False)
          (x_proj): Linear(in_features=1536, out_features=80, bias=False)
          (dt_proj): Linear(in_features=48, out_features=1536, bias=True)
          (out_proj): Linear(in_features=1536, out_features=768, bias=False)
        )
      )
    )
    (norm_f): MambaRMSNorm(768, eps=1e-05)
  )
  (lm_head): Linear(in_features=768, out_features=50280, bias=False)
)

# Load checkpoint from training

In [ ]:
from mamba_ssm import MambaLMHeadModel

test = MambaLMHeadModel(
    config=
)

In [17]:
test

Mamba(
  (in_proj): Linear(in_features=768, out_features=3072, bias=False)
  (conv1d): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), padding=(3,), groups=1536)
  (act): SiLU()
  (x_proj): Linear(in_features=1536, out_features=80, bias=False)
  (dt_proj): Linear(in_features=48, out_features=1536, bias=True)
  (out_proj): Linear(in_features=1536, out_features=768, bias=False)
)

In [3]:
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM
from transformers.models.mamba.modeling_mamba import MambaForCausalLM, MambaCache, MambaModel, MambaConfig

model = MambaForCausalLM.from_pretrained(
    "../outputs/checkpoint-264970"
)




In [54]:
model.backbone.embeddings.weight

Parameter containing:
tensor([[-0.2310, -0.1452, -0.0195,  ..., -0.3611,  0.1751,  0.3919],
        [-0.2206, -0.1748, -0.0299,  ..., -0.3597,  0.1808,  0.3847],
        [ 0.1245,  0.0648,  0.0185,  ..., -0.0282,  0.1599,  0.3335],
        ...,
        [-0.2640, -0.1382, -0.0248,  ..., -0.3358,  0.1423,  0.3950],
        [-0.2242, -0.1338, -0.0395,  ..., -0.3347,  0.1349,  0.4008],
        [-0.2476, -0.1351, -0.0124,  ..., -0.3428,  0.1831,  0.3821]],
       requires_grad=True)

In [ ]:
import os

optimizer.pt
trainer_state.json
rng_state.pth
training_args.bin
config.json
scheduler.pt
generation_config.json
model.safetensors


In [46]:
model

MambaForCausalLM(
  (backbone): MambaModel(
    (embeddings): Embedding(50280, 768)
    (layers): ModuleList(
      (0-3): 4 x MambaBlock(
        (norm): MambaRMSNorm(768, eps=1e-05)
        (mixer): MambaMixer(
          (conv1d): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), padding=(3,), groups=1536)
          (act): SiLU()
          (in_proj): Linear(in_features=768, out_features=3072, bias=False)
          (x_proj): Linear(in_features=1536, out_features=80, bias=False)
          (dt_proj): Linear(in_features=48, out_features=1536, bias=True)
          (out_proj): Linear(in_features=1536, out_features=768, bias=False)
        )
      )
    )
    (norm_f): MambaRMSNorm(768, eps=1e-05)
  )
  (lm_head): Linear(in_features=768, out_features=50280, bias=False)
)

In [ ]:
# This is an unofficial implementation of Mamba
# However, it is very simple for us to get activations
# and weights
from mamba_py.mambapy.mamba import Mamba, MambaConfig, MambaForLM
from torch import nn
from transformers import AutoTokenizer



model = MambaForLM(config)


In [6]:
model

MambaForLM(
  (backbone): ModuleDict(
    (embeddings): Embedding(50280, 768)
    (layers): ModuleList(
      (0-3): 4 x ResidualBlock(
        (mixer): MambaBlock(
          (in_proj): Linear(in_features=768, out_features=3072, bias=False)
          (conv1d): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), padding=(3,), groups=1536)
          (x_proj): Linear(in_features=1536, out_features=80, bias=False)
          (dt_proj): Linear(in_features=48, out_features=1536, bias=True)
          (out_proj): Linear(in_features=1536, out_features=768, bias=False)
        )
        (norm): RMSNorm()
      )
    )
    (norm_f): RMSNorm()
  )
  (lm_head): Linear(in_features=768, out_features=50280, bias=False)
)

In [6]:
from mamba_py.mambapy.mamba import MambaForLM, MambaConfig

weights_path = "../outputs/checkpoint-264970/model.safetensors"

d_model = 768
n_layers = 4
num_tokens = 50280 # taken from HF
config = MambaConfig(d_model, n_layers, num_tokens)

model = MambaForLM.from_pretrained_safetensors(weights_path, config)

Tied weights: Copying embeddings weights to final head


In [7]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "state-spaces/mamba-130m-hf"
)

# Test the model
prompt = "Once upon a time"
input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].cuda()
model = model.cuda()

outs = model(input_ids)

In [ ]:
model.cache[0]['inputs']

torch.Size([1, 4, 768])

In [19]:
out_mamba_block = model.backbone.layers[0].mixer.out_proj(model.cache[0]['outputs'] * model.cache[0]['y'])
out_mamba_block + model.cache[0]['pre_rms']

tensor([[[  4.9237,  -6.7575,  -1.4425,  ...,  15.3584,   3.9913,   6.8482],
         [ 75.7084,  62.4738, -11.5577,  ..., 108.1012, -13.1895,  -3.6021],
         [-30.9738,  13.0468,   6.7677,  ...,  13.7131, -22.3424, -24.7860],
         [ -9.6751, -20.5682,  -0.3765,  ...,  39.4694,  -1.7126,   1.8016]]],
       device='cuda:0', grad_fn=<AddBackward0>)

In [17]:
model.cache[0]['out_block']

tensor([[[  4.9237,  -6.7575,  -1.4425,  ...,  15.3584,   3.9913,   6.8482],
         [ 75.7084,  62.4738, -11.5577,  ..., 108.1012, -13.1895,  -3.6021],
         [-30.9738,  13.0468,   6.7677,  ...,  13.7131, -22.3424, -24.7860],
         [ -9.6751, -20.5682,  -0.3765,  ...,  39.4694,  -1.7126,   1.8016]]],
       device='cuda:0', grad_fn=<AddBackward0>)

In [11]:
# Get RMSNorm constant for layer l for token t
l = 1
t = 3

post_norm = model.cache[l]["inputs"][:, t]
pre_norm = model.cache[l]["pre_rms"][:, t]

(post_norm / pre_norm).shape

torch.Size([1, 768])

In [4]:
from mambacoder import MambaCoder
from config import TranscoderConfig

t_cfg = TranscoderConfig(weight_path="../outputs/checkpoint-264970/model.safetensors", save_path="../models/MambaCoders/mambacoder_1536.pt")

mc = MambaCoder.load_model(
    t_cfg,
    config
)

Tied weights: Copying embeddings weights to final head


In [ ]:
import torch
import torch.nn.functional as F

# Adapted from https://github.com/AmeenAli/HiddenMambaAttn/blob/main/mamba-1p1p1/mamba_ssm/ops/selective_scan_interface.py#L741
def compute_attn_matrix_fn(dA: torch.Tensor, dB: torch.Tensor, C: torch.Tensor):
    B, L, ED, N = A.shape
    attn_matrices = torch.zeros((B, ED, L, L), device=A.device) # B, ED, L, L
    for r in range(L):
        # This is the output idx
        # C is equal for each head, it depends solely on the output idx
        curr_C = C[:, r, :] # (B, N)
        for c in range(r+1):
            # Now store the As, we are going from c to r (c, c+1, c+2, ..., r)
            curr_A = torch.ones((B, ED, N), device=A.device)
            if c < r:
                for i in range(r-c):
                    curr_A = curr_A * dA[:, r-i, :, :]
            curr_B = dB[:, c, :, :]
            attn_matrices[:, :, r, c] = torch.sum(curr_C * curr_A * curr_B, axis=-1)
    return attn_matrices

A = model.backbone.layers[0].mixer.deltaA # (L, ED, N) because B is 1
B = model.backbone.layers[0].mixer.deltaB # (L, ED, N) because B is 1
C = model.backbone.layers[0].mixer.C # (B, L, N) because B is 1

attn_matrix = compute_attn_matrix_fn(A, B, C)

# Pseudo feature vector
feature_vector = torch.randn((model.config.expand_factor * model.config.d_model), device=A.device)

query_tok = 2
key_tok = 3
head = 512

attn_scores = attn_matrix[0, :, key_tok, query_tok]

inputs_layer_0 = model.cache[0]['inputs']
B, L, D = inputs_layer_0.shape
inputs_conv = model.backbone.layers[0].mixer.in_proj(inputs_layer_0)
inputs_conv = inputs_conv.chunk(2, dim=-1)[0]
outs_conv = model.backbone.layers[0].mixer.conv1d(inputs_conv.transpose(-1, -2))
outs_conv = outs_conv[:, :, :L].transpose(-1, -2)
outs_conv = F.silu(outs_conv) # (B, L, ED)

# Contribution from token s is 
# feature vector # (ED)

# want output to be (B, )


torch.Size([1, 4, 1536])

In [7]:
W2_SSM, W2_gate = model.backbone.layers[0].mixer.in_proj.weight.data.chunk(2, dim=-2)

In [11]:
import torch
ED = 1536
(torch.rand(ED, device=W2_SSM.device) @ W2_SSM).shape

torch.Size([768])

In [22]:
model

MambaForLM(
  (backbone): ModuleDict(
    (embeddings): Embedding(50280, 768)
    (layers): ModuleList(
      (0-3): 4 x ResidualBlock(
        (mixer): MambaBlock(
          (in_proj): Linear(in_features=768, out_features=3072, bias=False)
          (conv1d): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), padding=(3,), groups=1536)
          (x_proj): Linear(in_features=1536, out_features=80, bias=False)
          (dt_proj): Linear(in_features=48, out_features=1536, bias=True)
          (out_proj): Linear(in_features=1536, out_features=768, bias=False)
        )
        (norm): RMSNorm()
      )
    )
    (norm_f): RMSNorm()
  )
  (lm_head): Linear(in_features=768, out_features=50280, bias=False)
)

In [ ]:
D = 768
x = torch.ones(D).cuda()
rms_x = torch.sqrt(1e-05 + torch.sum(x ** 2) / D)

tensor(1.0000, device='cuda:0')

In [346]:
import torch.nn.functional as F

x = torch.randn(10)

print(F.silu(x))
print(F.sigmoid(x) * x)


tensor([ 0.6737, -0.2316, -0.2779, -0.2749, -0.1557, -0.2613, -0.2766,  0.6404,
        -0.2596,  0.6706])
tensor([ 0.6737, -0.2316, -0.2779, -0.2749, -0.1557, -0.2613, -0.2766,  0.6404,
        -0.2596,  0.6706])


In [337]:
attn_matrix.shape

torch.Size([1, 1536, 4, 4])

In [101]:
model.backbone.layers[0].mixer.conv1d.weight.data.shape

torch.Size([1536, 1, 4])

In [112]:
def construct_toeplitz(kernels: torch.Tensor, seq_len: int, stride: int = 1) -> torch.Tensor:
    # Conv weights are stored as C, B, K
    C, B, K = kernels.shape

    # Define the out shape
    out_len = K + seq_len - stride # here I'm assuming stride = 1

    # Define indices for mask
    i = torch.arange(out_len, device=kernels.device).unsqueeze(1)
    j = torch.arange(K, device=kernels.device).unsqueeze(0)

    # Calculate valid entries
    mask = ((i - j) >= 0) & ((i - j) < K)
    mask = mask.float()

    # Reverse the kernels (we have a convolution)
    k_rev = torch.flip(kernels, dims=(-1,))

    # Now define the toeplitz matrix
    toeplitz = torch.zeros_like(mask)
    idx = (i - j)
    toeplitz[mask.bool()] = k_rev[idx[mask.bool()]]

    return toeplitz


construct_toeplitz(model.backbone.layers[0].mixer.conv1d.weight.data, L)







RuntimeError: shape mismatch: value tensor of shape [16, 1, 4] cannot be broadcast to indexing result of shape [16]

In [160]:
import torch

def construct_toeplitz(kernels: torch.Tensor, seq_len: int, stride: int = 1) -> torch.Tensor:
    """
    Build per-channel Toeplitz matrices for 1D conv kernels.

    Args:
        kernels: (C, 1, K)
        seq_len: input sequence length (L)
        stride: convolution stride (default=1)

    Returns:
        toeplitz: (C, out_len, seq_len) — Toeplitz matrix per channel
    """
    C, _, K = kernels.shape
    device = kernels.device

    # Compute output length for stride=1 (same as conv1d output length formula)
    out_len = seq_len + K - 1
    # You can adjust this if you apply padding or truncation

    # Indices
    i = torch.arange(out_len, device=device).unsqueeze(1)  # (out_len, 1)
    j = torch.arange(seq_len, device=device).unsqueeze(0)  # (1, seq_len)

    # Compute offsets (how far each input index is from current output position)
    offset = i - j  # (out_len, seq_len)

    # Reverse kernels for convolution
    k_rev = torch.flip(kernels.squeeze(1), dims=[-1])  # (C, K)

    # Build mask of valid indices
    mask = (offset >= 0) & (offset < K)  # (out_len, seq_len)
    mask_expanded = mask.unsqueeze(0).expand(C, -1, -1)  # (C, out_len, seq_len)

    # Create tensor to hold results
    toeplitz = torch.zeros(C, out_len, seq_len, device=device, dtype=kernels.dtype)

    # Use broadcasting and advanced indexing to fill in values
    valid_offsets = offset.clamp(min=0, max=K-1)
    toeplitz[mask_expanded] = k_rev[:, valid_offsets[mask]].flatten()

    return toeplitz

toep = construct_toeplitz(model.backbone.layers[0].mixer.conv1d.weight.data, L)

# Now perform computation with the input
ED = 1536
pseudo_input = torch.rand((B, L, ED), device=toep.device)
pseudo_input = pseudo_input.transpose(-1, -2)

result = torch.einsum("COK, BCK -> BCO", toep, pseudo_input)
result

tensor([[[-0.0350, -0.1724, -0.0930,  ..., -0.1030, -0.0100,  0.0006],
         [ 0.0447, -0.0999,  0.3913,  ...,  0.4020,  0.4028,  0.2501],
         [ 0.0471,  0.1485,  0.0962,  ...,  0.1126,  0.0059, -0.0065],
         ...,
         [-0.0285, -0.0902, -0.1077,  ..., -0.0387, -0.0126, -0.0094],
         [-0.0379, -0.1191, -0.0723,  ...,  0.1562, -0.0197, -0.0042],
         [ 0.0106, -0.0161, -0.1292,  ..., -0.3049, -0.0712,  0.0093]]],
       device='cuda:0')

In [334]:
def conv2Mat(conv1d_weight, L):
    H, D = conv1d_weight.shape
    conv1d_weight = torch.flip(conv1d_weight,[-1])
    # Initialize the matrix with zeros
    M = torch.zeros(H, L, L).to(conv1d_weight.device)
    
    # Fill the matrix with the kernel weights
    for h in range(H):
        for i in range(L):
            # Set the diagonal and the next (D-1) positions, respecting the input length L
            M[h, i, i:i+D] = conv1d_weight[h, :max(0, D - (i + D - L))]
    return M.flip(dims=[-1])

L = 7
ED = 1536
pseudo_input = torch.rand((B, L, ED), device=toep.device)

M = conv2Mat(model.backbone.layers[0].mixer.conv1d.weight.data.squeeze(1), L)
bias = model.backbone.layers[0].mixer.conv1d.bias

# Now calculate out using M
conv1d_out_mat = torch.einsum("COL, BLC -> BOC", M, pseudo_input).flip(dims=[-2]) + bias

# Calculate using real conv
in_conv = pseudo_input.transpose(-1, -2)
out_conv = model.backbone.layers[0].mixer.conv1d(in_conv)
out_conv = out_conv[:, :, :L].transpose(-1, -2)

torch.testing.assert_close(conv1d_out_mat, out_conv)



In [329]:
kernel_idx = 4
out_conv[:, :, kernel_idx]

tensor([[0.3910, 0.8678, 0.8445, 0.6479, 0.4544, 0.3287, 0.4910]],
       device='cuda:0', grad_fn=<SelectBackward0>)

In [330]:
conv1d_out_mat[:, :, kernel_idx]

tensor([[0.2788, 0.3590, 0.1475, 0.2113, 0.4013, 0.3665, 0.0943]],
       device='cuda:0')

In [331]:
import numpy as np

kernel_idx = 10

channel = pseudo_input[:, :, kernel_idx].cpu()
kernel = model.backbone.layers[0].mixer.conv1d.weight.data[kernel_idx, :, :].cpu()

res = np.convolve(channel.squeeze(), kernel.flip(dims=(-1,)).squeeze()) 
print(res)

real = model.backbone.layers[0].mixer.conv1d(pseudo_input.transpose(-1, -2))
print(real.transpose(-1, -2)[0, :, kernel_idx].cpu() - model.backbone.layers[0].mixer.conv1d.bias.data[kernel_idx].cpu().item())

print((M[kernel_idx, :, :] @ pseudo_input[:, :, kernel_idx].squeeze()).flip(dims=[-1]))

print(torch.einsum("COL, BLC -> BOC", M, pseudo_input)[:, :, kernel_idx].flip(dims=[-1]))


[0.01557068 0.09552611 0.08263731 0.10963483 0.05807264 0.094541
 0.06548148 0.1110849  0.01002192 0.02813271]
tensor([0.0156, 0.0955, 0.0826, 0.1096, 0.0581, 0.0945, 0.0655, 0.1111, 0.0100,
        0.0281], grad_fn=<SubBackward0>)
tensor([0.0156, 0.0955, 0.0826, 0.1096, 0.0581, 0.0945, 0.0655],
       device='cuda:0')
tensor([[0.0156, 0.0955, 0.0826, 0.1096, 0.0581, 0.0945, 0.0655]],
       device='cuda:0')


In [282]:
M_flip = M.flip(dims=[-1])

In [290]:
M_flip.shape

torch.Size([1536, 7, 7])

In [300]:
x = pseudo_input.transpose(-1, -2)

In [314]:
torch.einsum("COL, BLC -> BOC", M_flip, pseudo_input)[:, :, kernel_idx].flip(dims=[-1])

tensor([[0.0183, 0.1135, 0.0869, 0.0346, 0.0645, 0.0841, 0.0947]],
       device='cuda:0')

In [289]:
(M_flip[kernel_idx, :, :] @ pseudo_input[:, :, kernel_idx].squeeze()).flip(dims=[-1])

tensor([0.0105, 0.0695, 0.0819, 0.0714, 0.0824, 0.0733, 0.0401],
       device='cuda:0')

In [267]:
pseudo_input[:, :, kernel_idx].squeeze()

tensor([0.5538, 0.7441, 0.4702, 0.5024, 0.5729, 0.1602, 0.5486],
       device='cuda:0')

In [252]:
M[kernel_idx, :, :]

tensor([[ 0.0302, -0.0026,  0.1001,  0.0189,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0302, -0.0026,  0.1001,  0.0189,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0302, -0.0026,  0.1001,  0.0189,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0302, -0.0026,  0.1001,  0.0189],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0302, -0.0026,  0.1001],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0302, -0.0026],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0302]],
       device='cuda:0')

In [254]:
M[kernel_idx, :, :]

tensor([[ 0.0189,  0.1001, -0.0026,  0.0302,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0189,  0.1001, -0.0026,  0.0302,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0189,  0.1001, -0.0026,  0.0302,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0189,  0.1001, -0.0026,  0.0302],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0189,  0.1001, -0.0026],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0189,  0.1001],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0189]],
       device='cuda:0')

In [211]:
kernel

tensor([[-0.0144,  0.0429,  0.1727,  0.0562]])

tensor([-0.1085, -0.2743, -0.2799, -0.2436, -0.2215, -0.3365, -0.3684, -0.3085,
        -0.0951, -0.0720], grad_fn=<ToCopyBackward0>)

In [190]:
channel

tensor([[0.5309, 0.8035, 0.5476, 0.7522, 0.0190, 0.2713, 0.1563]])

In [183]:
model.backbone.layers[0].mixer.conv1d.weight.data

tensor([[[ 0.0015, -0.0265, -0.2418, -0.0508]],

        [[ 0.4593,  0.4062, -0.1079,  0.0467]],

        [[-0.0144,  0.0429,  0.1727,  0.0562]],

        ...,

        [[-0.0569,  0.0144, -0.2264, -0.1140]],

        [[-0.0494,  0.2358, -0.1514, -0.1006]],

        [[ 0.0107, -0.0923, -0.2724,  0.0618]]], device='cuda:0')

In [184]:
M[-2]

tensor([[-0.1006, -0.1514,  0.2358, -0.0494,  0.0000,  0.0000,  0.0000],
        [ 0.0000, -0.1006, -0.1514,  0.2358, -0.0494,  0.0000,  0.0000],
        [ 0.0000,  0.0000, -0.1006, -0.1514,  0.2358, -0.0494,  0.0000],
        [ 0.0000,  0.0000,  0.0000, -0.1006, -0.1514,  0.2358, -0.0494],
        [ 0.0000,  0.0000,  0.0000,  0.0000, -0.1006, -0.1514,  0.2358],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000, -0.1006, -0.1514],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000, -0.1006]],
       device='cuda:0')

In [ ]:
x = pseudo_input.transpose(-1, -2)
torch.matmul()

torch.Size([1, 1536, 7])

In [181]:
my_res = torch.einsum('HLL, BLH -> BLH', M, pseudo_input)
my_res

tensor([[[-0.0352,  0.0237,  0.0203,  ..., -0.0259, -0.0534,  0.0263],
         [-0.0336,  0.0212,  0.0422,  ..., -0.1088, -0.0808,  0.0503],
         [-0.0283,  0.0463,  0.0409,  ..., -0.1107, -0.0551,  0.0462],
         ...,
         [-0.0433,  0.0101,  0.0033,  ..., -0.0239, -0.0019,  0.0596],
         [-0.0481,  0.0010,  0.0117,  ..., -0.0154, -0.0273,  0.0487],
         [-0.0444,  0.0197,  0.0425,  ..., -0.1133, -0.0157,  0.0603]]],
       device='cuda:0')

In [197]:
out = model.backbone.layers[0].mixer.conv1d(pseudo_input.transpose(-1, -2))
real = out[:, :, :L].transpose(-1, -2)
real

tensor([[[-0.1085, -0.3886,  0.2056,  ...,  0.0149, -0.4026, -0.0956],
         [-0.2743, -0.4456,  0.2898,  ..., -0.1194, -0.5104, -0.1875],
         [-0.2799, -0.2093,  0.3715,  ..., -0.2827, -0.4008, -0.3365],
         ...,
         [-0.2215,  0.1137,  0.2211,  ..., -0.0538, -0.3756, -0.1919],
         [-0.3365,  0.3781,  0.1997,  ..., -0.0754, -0.2291, -0.3514],
         [-0.3684,  0.0972,  0.2654,  ..., -0.1077, -0.4387, -0.3625]]],
       device='cuda:0', grad_fn=<TransposeBackward0>)

In [177]:
out.transpose(-1, -2)[0, :, -1]

tensor([-0.0977, -0.1969, -0.2890, -0.1796, -0.1983, -0.2449, -0.3316, -0.3486,
        -0.1690, -0.1155], device='cuda:0', grad_fn=<SelectBackward0>)

In [159]:
model.backbone.layers[0].mixer.conv1d.bias.data

tensor([-0.0733, -0.4122,  0.1853,  ...,  0.0408, -0.3492, -0.1219],
       device='cuda:0')

In [154]:
real - my_res

tensor([[[-0.0733, -0.4122,  0.1853,  ...,  0.0408, -0.3492, -0.1219],
         [-0.0912, -0.4230,  0.3287,  ..., -0.0376, -0.4014, -0.2286],
         [-0.1131, -0.4151,  0.3620,  ..., -0.1386, -0.2803, -0.2975],
         ...,
         [-0.1585, -0.1942,  0.3229,  ..., -0.1282, -0.3341, -0.2293],
         [-0.1995, -0.1976,  0.3008,  ..., -0.1802, -0.2922, -0.2912],
         [-0.2043, -0.1755,  0.2503,  ..., -0.0441, -0.3131, -0.3685]]],
       device='cuda:0', grad_fn=<SubBackward0>)

In [ ]:
B, L, ED = 1, 4, 1536

pseudo_input = torch.rand((B, L, ED), device=A.device)
first_channel_kernel = model.backbone.layers[0].mixer.conv1d.weight.data[0, :, :].squeeze()
first_channel_bias = model.backbone.layers[0].mixer.conv1d.bias[0].cpu().item()

pseudo_input_first_channel = pseudo_input[:, :, 0]

out_conv_real = model.backbone.layers[0].mixer.conv1d(pseudo_input.transpose(-1, -2))

# Define the conv toeplitz matrix
K = first_channel_kernel.numel()
out_len = K + L - 1
i = torch.arange(out_len).unsqueeze(1)  # (L, 1)
j = torch.arange(K).unsqueeze(0)  # (1, K)

# Valid entries correspond to places where i-j is in [0, K-1]
mask = ((i - j) >= 0) & ((i - j) < K)
mask = mask.float()  # 1.0 for valid, 0.0 for zero positions

k_rev = torch.flip(first_channel_kernel, dims=[0]).cpu()

toeplitz = torch.zeros_like(mask)
idx = (i - j)
toeplitz[mask.bool()] = k_rev[idx[mask.bool()]]

res = torch.einsum('LK, K -> L', toeplitz, pseudo_input_first_channel.cpu().squeeze()) + first_channel_bias

print("Reconstructed output:")
print(res)
print("Real output")
print(out_conv_real[0, 0, :])
print(f"Difference: {res - out_conv_real[0, 0, :].cpu()}")



Reconstructed output:
tensor([-7.3252e-03, -6.5635e-02, -1.8170e-01, -1.6603e-01, -1.9018e-02,
         5.4822e-04,  2.2334e-05])
Real output
tensor([-0.0806, -0.1390, -0.2550, -0.2394, -0.0923, -0.0728, -0.0733],
       device='cuda:0', grad_fn=<SelectBackward0>)
Difference: tensor([0.0733, 0.0733, 0.0733, 0.0733, 0.0733, 0.0733, 0.0733],
       grad_fn=<SubBackward0>)


In [106]:
model.backbone.layers[0].mixer.conv1d.weight.data

tensor([[[ 0.0015, -0.0265, -0.2418, -0.0508]],

        [[ 0.4593,  0.4062, -0.1079,  0.0467]],

        [[-0.0144,  0.0429,  0.1727,  0.0562]],

        ...,

        [[-0.0569,  0.0144, -0.2264, -0.1140]],

        [[-0.0494,  0.2358, -0.1514, -0.1006]],

        [[ 0.0107, -0.0923, -0.2724,  0.0618]]], device='cuda:0')

In [109]:
torch.flip(model.backbone.layers[0].mixer.conv1d.weight.data, dims=(-1,))

tensor([[[-0.0508, -0.2418, -0.0265,  0.0015]],

        [[ 0.0467, -0.1079,  0.4062,  0.4593]],

        [[ 0.0562,  0.1727,  0.0429, -0.0144]],

        ...,

        [[-0.1140, -0.2264,  0.0144, -0.0569]],

        [[-0.1006, -0.1514,  0.2358, -0.0494]],

        [[ 0.0618, -0.2724, -0.0923,  0.0107]]], device='cuda:0')

In [49]:
import torch
from circuit_tracing import FeatureVector, Component
from circuit_tracing import FeatureType, ComponentType
from torch.nn.functional import sigmoid

layer = 2
token = 3
k = 5

acts = model.cache[layer]["inputs"][..., token, :].squeeze()

# Now calculate features
mc_acts = mc.activation_functions[layer](mc.encoders[layer](acts))

# Get features different from zero
_, act_features_idx = torch.topk(mc_acts, k = mc.cfg.topk_features)

# Take the first one
most_act_feature = act_features_idx[0]
print(f"Most activated feature in layer {layer} with act={mc_acts[most_act_feature]:.4f} is feature no. {most_act_feature}")

# Define the corresponding feature vector
feature_vector = mc.encoders[layer].weight[most_act_feature, :] # f_enc(l, i) where i=most_act_feature

# The first thing to do is to trace back this feature vector
# that lives in the residual stream into the 
# Mamba block space
W3 = mc.base_model.backbone.layers[layer].mixer.out_proj.weight
feature_vector = W3.T @ feature_vector
# now this vector lives in the mamba block

# Now we need to trace it back

for l in range(layer):
    
    print(f"Layer {l}: Checking for gate path...")

    # I know that feature feat is active in layer l on token t
    # So my x = acts
    # Here we are considering the TC in layer l
    pulledback_features = mc.decoders[l].weight.T @ feature_vector # (num_features, ED) x (ED, 1)

    x = model.cache[l]["inputs"][..., token, :].squeeze()
    feature_activs = mc.encoders[l](x) # (num_features)
   
    # Now calculate the contribution
    contribs = pulledback_features * feature_activs # (num_features)

    # Take top 5 contributions
    top_contribs, top_idxs = torch.topk(pulledback_features, k=k)

    top_contrib_list = []

    print(f"In layer {l} the top contributions come from features {top_idxs.cpu().numpy().tolist()}")

    # Rebuild feature vector objs from features
    for contrib, idx in zip(top_contribs, top_idxs):
        vector = mc.encoders[l].weight[idx, :] # (D)
        vector = vector * (mc.decoders[layer].weight.T @ feature_vector)[idx] # (is D dimensional)

        print(f"{idx}: Contrib={contrib.item():.2f}")

        # We are computing f = (f_enc * f_dec) * f
        new_component = Component(
            layer=l,
            component_type=ComponentType.GATE,
            token=token,
            feature_type=FeatureType.MAMBACODER,
            feature_idx=idx.item() # New component from index idx in layer l
        )

        top_contrib_list.append(FeatureVector(
            component_path=[new_component],
            vector=vector,
            layer=l,
            sublayer="resid_pre",
            contrib=contrib.item()
        ))

    print(top_contrib_list) 
    
    

Most activated feature in layer 2 with act=2.8423 is feature no. 10089
Layer 0: Checking for gate path...
In layer 0 the top contributions come from features [3098, 1438, 9912, 4107, 11583]
3098: Contrib=5.35
1438: Contrib=4.38
9912: Contrib=3.37
4107: Contrib=3.15
11583: Contrib=3.06
[<FeatureVector object gate0mc[3098]@3: 5.4, sublayer=resid_pre>, <FeatureVector object gate0mc[1438]@3: 4.4, sublayer=resid_pre>, <FeatureVector object gate0mc[9912]@3: 3.4, sublayer=resid_pre>, <FeatureVector object gate0mc[4107]@3: 3.2, sublayer=resid_pre>, <FeatureVector object gate0mc[11583]@3: 3.1, sublayer=resid_pre>]
Layer 1: Checking for gate path...
In layer 1 the top contributions come from features [4615, 8914, 8732, 4631, 8483]
4615: Contrib=2.82
8914: Contrib=2.81
8732: Contrib=2.69
4631: Contrib=2.60
8483: Contrib=2.59
[<FeatureVector object gate1mc[4615]@3: 2.8, sublayer=resid_pre>, <FeatureVector object gate1mc[8914]@3: 2.8, sublayer=resid_pre>, <FeatureVector object gate1mc[8732]@3: 2.7,